In [27]:
from mne.decoding import CSP
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from src.classifier import LDA
from sklearn.metrics import accuracy_score
import pandas as pd

subjects = list(range(1, 11))

base = Path("./Dataset") / "Lee2019_MI"

EPOCH_WINDOWS = [(0, 3), (0.5, 3.5)]
EPOCH_WINDOWS = [(0, 3)]


In [28]:
pipeline = Pipeline([("csp", CSP(n_components=6)), ("lda", LDA())])

In [38]:
results_cs = {"subject": [], "accuracy": [], "tmin": [], "tmax": []}
acc_list = []
for tmin, tmax in EPOCH_WINDOWS:
    suffix = f"{tmin}_{tmax}"
    for subject in subjects:
        X_train = np.load(base / "per_session" / f"sub-{subject}_ses-1_run-1_X_{suffix}.npy")
        y_train = np.load(base / "per_session" / f"sub-{subject}_ses-1_run-1_y_{suffix}.npy")

        X_test = np.load(base / "per_session" / f"sub-{subject}_ses-2_run-1_X_{suffix}.npy")
        y_test = np.load(base / "per_session" / f"sub-{subject}_ses-2_run-1_y_{suffix}.npy")

        csp = CSP(n_components=6)
        model = LDA()

        model.fit(csp.fit_transform(X_train, y_train), y_train)

        preds = model.predict(csp.transform(X_test))

        acc = accuracy_score(y_test, preds)

        results_cs["subject"].append(subject)
        results_cs["accuracy"].append(acc)
        results_cs["tmin"].append(tmin)
        results_cs["tmax"].append(tmax)

Computing rank from data with rank=None
    Using tolerance 0.0034 (2.2e-16 eps * 62 dim * 2.5e+11  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.0008 (2.2e-16 eps * 62 dim * 5.8e+10  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.003 (2.2e-16 eps * 62 dim * 2.2e+11  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance using EMPIRIC

In [39]:
df_cs = pd.DataFrame(results_cs)

display(df_cs)

df_cs["accuracy"].mean()



,subject,accuracy,tmin,tmax
0,1,0.50,0,3
1,2,0.44,0,3
2,3,0.50,0,3
3,4,0.50,0,3
4,5,0.46,0,3
5,6,0.50,0,3
6,7,0.52,0,3
7,8,0.46,0,3
8,9,0.51,0,3
9,10,0.57,0,3


np.float64(0.496)

In [57]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

results_ws = {"subject": [], "accuracy": [], "tmin": [], "tmax": []}
acc_list = []
for tmin, tmax in EPOCH_WINDOWS:
    suffix = f"{tmin}_{tmax}"
    for subject in subjects:
        train_list = [1, 2, 3, 4, 5]
        test_list = [6, 7, 8, 9, 10]

        X_list, y_list = [], []
        for run_num in train_list:
            X = np.load(base / f"sub-{subject}_ses-1_run-{run_num}_X_{suffix}.npy")
            y = np.load(base / f"sub-{subject}_ses-1_run-{run_num}_y_{suffix}.npy")

            X_list.append(X)
            y_list.append(y)
        X_train = np.concatenate(X_list)
        y_train = np.concatenate(y_list)

        X_list, y_list = [], []
        for run_num in test_list:
            X = np.load(base / f"sub-{subject}_ses-1_run-{run_num}_X_{suffix}.npy")
            y = np.load(base / f"sub-{subject}_ses-1_run-{run_num}_y_{suffix}.npy")

            X_list.append(X)
            y_list.append(y)
        X_test = np.concatenate(X_list)
        y_test = np.concatenate(y_list)

        csp = CSP(n_components=6)
        model = LDA()

        model.fit(csp.fit_transform(X_train, y_train), y_train)

        preds = model.predict(csp.transform(X_test))

        acc = accuracy_score(y_test, preds)
        print(acc)

        results_ws["subject"].append(subject)
        results_ws["accuracy"].append(acc)
        results_ws["tmin"].append(tmin)
        results_ws["tmax"].append(tmax)



Computing rank from data with rank=None
    Using tolerance 0.0024 (2.2e-16 eps * 62 dim * 1.7e+11  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
0.46
Computing rank from data with rank=None
    Using tolerance 0.00055 (2.2e-16 eps * 62 dim * 4e+10  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
0.58
Computing rank from data with rank=None
    Using tolerance 0.002 (2.2e-16 eps * 62 dim * 1.5e+11  max singular value)
    Estimated rank (data): 62
    data: rank 62 computed from 62 data channels with 0 projectors
Reducing data rank from 62 -> 62
Estimating class=0 covariance usin

In [59]:
df_ws = pd.DataFrame(results_ws)

display(df_ws)

df_ws["accuracy"].mean()

,subject,accuracy,tmin,tmax
0,1,0.46,0,3
1,2,0.58,0,3
2,3,0.52,0,3
3,4,0.52,0,3
4,5,0.48,0,3
5,6,0.66,0,3
6,7,0.52,0,3
7,8,0.50,0,3
8,9,0.40,0,3
9,10,0.42,0,3


np.float64(0.506)

In [60]:
def compute_statistical_chance_level(n_samples, n_classes, alpha=0.05):
    from scipy.stats import binom

    k = binom.ppf(1 - alpha, n_samples, 1 / n_classes)
    k = k / n_samples

    return k


print(compute_statistical_chance_level(100, 2, 0.05))
print(compute_statistical_chance_level(50, 2, 0.05))
print(compute_statistical_chance_level(20, 2, 0.05))


0.58
0.62
0.7
